ワオキツネザル動画フィルタリング

条件：気温 ≥30.3℃（上位10%）× 風速 ≤2.3m/s（下位50%）

出力：該当動画リストをCSVに保存（コピーは行わない）
 
使用データ：
  - 気象CSV:kyototenki.csv
  - 日の出・日の入りCSV:KyotoHinode04/05/06.csv
  - 動画フォルダ: D:\ワオキツネザルCAM1
  - 出力先: C:\thesis\filtered_videos_final.csv

In [20]:
import pandas as pd
from datetime import datetime, timedelta
import os

パス設定

In [21]:
WEATHER_CSV = r"f:\test_cuts\kyototenki.csv"
HINODE_CSVS = [
    (4, r"F:\thesis\KyotoHinode04.csv"),
    (5, r'F:\thesis\KyotoHinode05.csv'),
    (6, r'F:\thesis\KyotoHinode06.csv'),
]
SOURCE_ROOT = r"f:\ワオキツネザルCAM1"
OUTPUT_CSV  = r'F:\thesis\filtered_videos_final.csv'

天気データ読み込み

In [24]:
raw  = open(WEATHER_CSV, 'rb').read()
text = raw.decode('cp932', errors='replace')
lines = text.split('\r\n')
 
# 最初の6行はヘッダー・メタ情報なのでスキップ
data_lines = [l for l in lines[6:] if l.strip()]
 
records = []
for line in data_lines:
    cols = line.split(',')
    if len(cols) < 3:
        continue
    try:
        dt   = datetime.strptime(cols[0].strip(), '%Y/%m/%d %H:%M:%S')
        temp = float(cols[1].strip())
        wind = float(cols[4].strip())
        records.append({'datetime': dt, 'temp': temp, 'wind': wind})
    except Exception:
        continue
 
weather_df = pd.DataFrame(records)
print(f"天気データ読み込み完了: {len(weather_df)}件")
 

天気データ読み込み完了: 1655件


日の出・日の入りデータ読み込み
CSVの構造：

#行0 → 「2025年 4月」などの月ヘッダー（スキップ）

#行1 → 「日・出・方位・南中・高度・入り・方位」（スキップ）

#行2以降 → 実データ（日・出時刻・方位・南中・高度・入り時刻・方位）

In [25]:
# 対象年（気象データの年に合わせる）
YEAR = 2025
 
hinode_records = []
for month, fpath in HINODE_CSVS:
    df_h = pd.read_csv(
        fpath,
        encoding='utf-8-sig',
        header=None,       # 自動ヘッダー判定を無効化
        skiprows=2,        # 月ヘッダー行・列名行の2行をスキップ
    )
    # 列の意味：0=日, 1=日の出時刻, 2=日の出方位, 3=南中, 4=南中高度, 5=日の入り時刻, 6=日の入り方位
    for _, row in df_h.iterrows():
        try:
            day       = int(row[0])
            rise_str  = str(row[1]).strip()   # 例："5:44"
            set_str   = str(row[5]).strip()   # 例："18:18"
 
            date_str  = f"{YEAR}/{month:02d}/{day:02d}"
            rise_time = datetime.strptime(f"{date_str} {rise_str}", '%Y/%m/%d %H:%M')
            set_time  = datetime.strptime(f"{date_str} {set_str}",  '%Y/%m/%d %H:%M')
 
            hinode_records.append({
                'date'   : datetime(YEAR, month, day).date(),
                'sunrise': rise_time,
                'sunset' : set_time,
            })
        except Exception:
            continue
 
hinode_df = pd.DataFrame(hinode_records)
print(f"✅ 日の出・日没データ読み込み完了: {len(hinode_df)}件")

✅ 日の出・日没データ読み込み完了: 91件


もしエンコーディングでエラーが出る場合↓

In [9]:
# エンコーディングを総当たりで試す
for month, fpath in HINODE_CSVS:
    for enc in ['utf-8-sig', 'utf-8', 'cp932', 'shift_jis']:
        try:
            df_test = pd.read_csv(fpath, encoding=enc, nrows=3)
            print(f"{fpath} → '{enc}' で読めました")
            print(df_test.head(3))
            break
        except Exception as e:
            print(f"  {enc}: NG ({e})")
    break  # 1ファイルだけ確認

F:\thesis\KyotoHinode04.csv → 'utf-8-sig' で読めました
  2025年 4月 Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6
0        日          出      方位[°]         南中      高度[°]         入り      方位[°]
1        1       5:44       83.9      12:01       59.6      18:18      276.3
2        2       5:43       83.4      12:01         60      18:19      276.8


夜間のデータを除外

In [26]:
hinode_dict = {row['date']: row for _, row in hinode_df.iterrows()}
 
def is_daytime(row):
    d = row['datetime'].date()
    h = hinode_dict.get(d)
    if h is None:
        return False
    return h['sunrise'] <= row['datetime'] <= h['sunset']
 
weather_df['is_day'] = weather_df.apply(is_daytime, axis=1)
daytime_df = weather_df[weather_df['is_day']].copy()
print(f"✅ 昼間データ: {len(daytime_df)}件")

✅ 昼間データ: 990件


気温と風速のパーセンタイルを把握

In [27]:
# 昼間データの気温・風速の分布を確認
print("=== 気温 ===")
print(daytime_df['temp'].describe())
print(f"\n上位10%の閾値: {daytime_df['temp'].quantile(0.90):.1f}℃")
print(f"上位25%の閾値: {daytime_df['temp'].quantile(0.75):.1f}℃")

print("\n=== 風速 ===")
print(daytime_df['wind'].describe())
print(f"\n下位50%の閾値: {daytime_df['wind'].quantile(0.50):.1f}m/s")
print(f"下位25%の閾値: {daytime_df['wind'].quantile(0.25):.1f}m/s")

print(f"\n昼間データ件数: {len(daytime_df)}件")
print(f"期間: {daytime_df['datetime'].min()} 〜 {daytime_df['datetime'].max()}")

=== 気温 ===
count    990.000000
mean      23.195859
std        5.252330
min        7.900000
25%       19.600000
50%       23.000000
75%       26.600000
max       36.500000
Name: temp, dtype: float64

上位10%の閾値: 30.3℃
上位25%の閾値: 26.6℃

=== 風速 ===
count    990.000000
mean       2.455960
std        1.280656
min        0.000000
25%        1.400000
50%        2.300000
75%        3.300000
max        7.200000
Name: wind, dtype: float64

下位50%の閾値: 2.3m/s
下位25%の閾値: 1.4m/s

昼間データ件数: 990件
期間: 2025-04-23 06:00:00 〜 2025-06-30 19:00:00


気象条件で絞り込み

気温 ≥30.3℃（上位10%）× 風速 ≤2.3m/s（下位50%）

In [28]:
TEMP_THRESHOLD = 30.3
WIND_THRESHOLD =  2.3
 
filtered = daytime_df[
    (daytime_df['temp'] >= TEMP_THRESHOLD) &
    (daytime_df['wind'] <= WIND_THRESHOLD)
].copy()
 
print(f"✅ 条件合致: {len(filtered)}時間 / 推定約{len(filtered)*2}本")
print(filtered[['datetime','temp','wind']].to_string(index=False))

✅ 条件合致: 25時間 / 推定約50本
           datetime  temp  wind
2025-06-13 14:00:00  30.3   1.5
2025-06-18 10:00:00  30.3   1.7
2025-06-18 12:00:00  33.5   2.3
2025-06-18 18:00:00  32.6   2.3
2025-06-18 19:00:00  31.7   1.7
2025-06-19 10:00:00  31.1   2.2
2025-06-20 14:00:00  35.7   2.2
2025-06-21 12:00:00  31.6   2.0
2025-06-22 11:00:00  30.6   1.8
2025-06-22 19:00:00  30.5   2.3
2025-06-25 13:00:00  31.2   1.9
2025-06-25 16:00:00  31.6   2.2
2025-06-27 13:00:00  31.2   1.8
2025-06-27 14:00:00  31.2   1.8
2025-06-27 16:00:00  31.5   1.6
2025-06-28 12:00:00  31.5   1.7
2025-06-28 13:00:00  32.3   1.8
2025-06-28 16:00:00  32.7   2.3
2025-06-28 17:00:00  33.1   1.6
2025-06-29 10:00:00  30.3   1.5
2025-06-29 11:00:00  31.3   1.8
2025-06-29 12:00:00  33.0   2.3
2025-06-29 13:00:00  33.7   2.1
2025-06-30 10:00:00  31.8   1.7
2025-06-30 11:00:00  33.4   1.8


動画ファイルと照合

In [29]:
target_videos = []
 
for _, row in filtered.iterrows():
    dt = row['datetime']
    for root, dirs, files in os.walk(SOURCE_ROOT):
        for fname in files:
            if not fname.lower().endswith(('.mp4', '.avi', '.mov')):
                continue
            # ファイル名から日時部分を探す（YYYYMMDD と HHmmss が隣り合っている前提）
            parts = fname.replace('-', '_').split('_')
            for i, p in enumerate(parts):
                if len(p) == 8 and p.isdigit() and i + 1 < len(parts):
                    time_part = parts[i+1][:6]
                    if len(time_part) == 6 and time_part.isdigit():
                        try:
                            file_dt = datetime.strptime(p + time_part, '%Y%m%d%H%M%S')
                            diff = abs((file_dt - dt).total_seconds())
                            if diff <= 1800:  # ±30分以内
                                target_videos.append({
                                    'datetime'     : dt,
                                    'temp'         : row['temp'],
                                    'wind'         : row['wind'],
                                    'folder'       : root,
                                    'filename'     : fname,
                                    'file_datetime': file_dt,
                                })
                        except Exception:
                            pass
 
print(f"✅ 照合できた動画: {len(target_videos)}本")

✅ 照合できた動画: 50本


csv出力

In [30]:
if target_videos:
    result_df = (
        pd.DataFrame(target_videos)
        [['datetime','temp','wind','folder','filename','file_datetime']]
        .sort_values('datetime')
        .drop_duplicates(subset='filename')
    )
    result_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n🎉 完了！{OUTPUT_CSV} に {len(result_df)} 本のリストを出力しました")
else:
    # 動画照合ができなくても気象条件の結果だけ保存
    filtered[['datetime','temp','wind']].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n⚠️  動画ファイルとの照合はゼロ件でした。")
    print(f"   気象条件の一覧だけ {OUTPUT_CSV} に出力しました。")
    print(f"   SOURCE_ROOT のパスを確認してください: {SOURCE_ROOT}")


🎉 完了！F:\thesis\filtered_videos_final.csv に 50 本のリストを出力しました
